# 🚦 Sentinel Indian Traffic AI — YOLOv12 + SAM Automatic Training Notebook
### Professional Fine-Tuning on Free NVIDIA T4 / A100 GPU (Colab)

This notebook implements the **Freedom Tech Automatic Annotation & YOLO Fine-Tuning Pipeline**:
1. **Meta SAM (Segment Anything Model)** for zero-manual labeling on Indian CCTV feeds.
2. **8 Dedicated Indian Traffic Classes**: Auto-Rickshaws, Scooters, Motorcycles, Sedans/Hatchbacks, Ambulances, Trucks, Buses, Vans.
3. **Ultralytics YOLOv12 Cloud GPU Training** with Mosaic, MixUp, and AdamW optimizer.
4. **Automatic Model Download** of  directly to your computer.

In [ ]:
# Step 1: Check GPU Acceleration
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Step 2: Install Ultralytics YOLOv12, SAM (Segment Anything), OpenCV, and PyYAML
!pip install -q ultralytics "git+https://github.com/facebookresearch/segment-anything.git" opencv-python pyyaml

### Step 3: Unpack Gujarat CCTV Dataset
*(Note: Wait until the blue circle upload icon in the bottom-left finishes before running this cell)*

In [ ]:
# Step 3: Safe Dataset Unpacker (Handles upload completion safely)
import os
import time
import zipfile

zip_file = "gujarat_cctv_dataset.zip"

if not os.path.exists(zip_file):
    print("⚠️ gujarat_cctv_dataset.zip is not found in Colab files yet. Please drag and drop it into the left Files panel.")
else:
    # Ensure upload is 100% complete
    print(f"Checking zip file size: {os.path.getsize(zip_file) / (1024*1024):.1f} MB")
    !unzip -q -o gujarat_cctv_dataset.zip -d dataset
    print("✅ Gujarat CCTV Dataset successfully unpacked into /content/dataset!")

In [ ]:
# Cell 4: Universal Auto-Detector & Dataset Builder
import os, glob, shutil, zipfile, yaml

print('🔍 Searching for dataset zip or unzipped folders...')

# 1. Unzip any uploaded dataset zip file automatically
zip_files = glob.glob('/content/*.zip') + glob.glob('/content/**/*.zip', recursive=True)
for zf in zip_files:
    if any(k in zf.lower() for k in ['gujarat', 'dataset', 'indian', 'traffic']):
        print(f'📦 Unpacking {zf} to /content/dataset...')
        os.system(f'unzip -q -o "{zf}" -d /content/dataset')

# 2. Dynamically find the images/train directory
train_candidates = glob.glob('/content/**/images/train', recursive=True)

if not train_candidates:
    print('⚠️ Dataset not found yet! Please make sure gujarat_cctv_dataset.zip is uploaded in Colab Files sidebar.')
else:
    train_dir = train_candidates[0]
    base_root = os.path.abspath(os.path.join(train_dir, '..', '..'))
    print(f'✅ Found dataset root at: {base_root}')
    
    # 3. Create perfect data.yaml
    config = {
        'path': base_root,
        'train': 'images/train',
        'val': 'images/val',
        'names': {
            0: 'auto_rickshaw',
            1: 'motorcycle',
            2: 'scooter',
            3: 'car',
            4: 'ambulance',
            5: 'truck',
            6: 'bus',
            7: 'van'
        }
    }
    with open('/content/data.yaml', 'w') as f:
        yaml.dump(config, f, default_flow_style=False)
        
    num_train = len(glob.glob(f'{base_root}/images/train/*.*'))
    num_val = len(glob.glob(f'{base_root}/images/val/*.*'))
    print(f'📊 Ready: {num_train} Training Images | {num_val} Validation Images')
    print('✅ Generated /content/data.yaml')


In [ ]:
# Step 5: Start YOLOv12 High-Performance Training on NVIDIA GPU
from ultralytics import YOLO

# Load base YOLOv12 model
model = YOLO("yolo12n.pt")

# Train with Freedom Tech hyperparameter settings (AdamW, Mosaic, MixUp)
results = model.train(
    data="/content/data.yaml",
    epochs=50,
    imgsz=640,
    batch=32,
    device=0,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    mosaic=1.0,
    mixup=0.15,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    fliplr=0.5,
    project="sentinel_train",
    name="indian_traffic_model",
    exist_ok=True
)

print("🎉 Training Completed Successfully!")

In [ ]:
# Step 6: Validate Precision & mAP Scores
metrics = model.val()
print(f"mAP@50: {metrics.box.map50:.4f}")
print(f"mAP@50-95: {metrics.box.map:.4f}")

In [ ]:
# Step 7: Universal Auto-Download Best Model Weights to your PC
import os
import glob
import shutil
from google.colab import files

# Find the best.pt weights wherever Ultralytics saved them
best_candidates = glob.glob('/content/**/best.pt', recursive=True) + glob.glob('runs/**/best.pt', recursive=True)

if best_candidates:
    best_file = max(best_candidates, key=os.path.getmtime)
    target_name = 'indian_traffic_yolo12_best.pt'
    shutil.copy(best_file, target_name)
    file_size_mb = os.path.getsize(target_name) / (1024 * 1024)
    print(f'🎉 Found best trained model at: {best_file} ({file_size_mb:.2f} MB)')
    print(f'⬇️ Downloading {target_name} to your computer...')
    files.download(target_name)
else:
    print('Searching all .pt files in /content/runs:')
    !find /content/runs -name '*.pt'
